# 🎙️ Hindi & Urdu Speech-to-Text  —  *“Whisper, but for Hindi/Urdu”*

Just like **Whisper turns English speech into text**, this notebook sets up a model
that **listens to Hindi and Urdu audio and writes out what was said** — this task is
called **Automatic Speech Recognition (ASR)**.

### The key idea
OpenAI's **Whisper is already multilingual.** The *same* model that transcribes English
natively understands **Hindi (`hi`)** and **Urdu (`ur`)** — plus ~97 other languages.
We simply tell it which language to listen for. Below we prove that, then use it, and
finally swap in models **fine-tuned specifically on Hindi/Urdu** for the best accuracy.

| # | Approach | Model | Best for |
|---|----------|-------|----------|
| 1 | **OpenAI Whisper** (official `whisper` package) | `whisper` (tiny→large-v3) | Fastest way to start — one line to transcribe |
| 2 | **Whisper via 🤗 Transformers** | `openai/whisper-large-v3` | Long audio, timestamps, production pipelines |
| 3 | **Fine-tuned Hindi / Urdu models** | `vasista22/...`, `kingabzpro/...` | **Highest accuracy** on Hindi/Urdu specifically |

> 💡 **Use a GPU** for real-time speed. In Colab: *Runtime → Change runtime type → T4 GPU*.
> Everything also runs on CPU, just slower.

## 1️⃣  Install dependencies

Whisper needs **`ffmpeg`** to decode audio, plus a few Python packages.
(`gTTS` is only used so this notebook can generate its own sample clips.)

In [ ]:
# ffmpeg = audio decoder (needed by Whisper). Works on Colab / Ubuntu.
!apt-get -qq update && apt-get -qq install -y ffmpeg

# Python packages
!pip install -q openai-whisper transformers accelerate gTTS

### ✅ Proof that Whisper understands Hindi & Urdu
Whisper ships with a built-in table of every language it was trained on.
Let's confirm Hindi and Urdu are in it.

In [ ]:
from whisper.tokenizer import LANGUAGES

for code_ in ['en', 'hi', 'ur']:
    print(f"{code_}  ->  {LANGUAGES[code_]}")

print('\nTotal languages Whisper supports:', len(LANGUAGES))

## 2️⃣  Get some Hindi / Urdu audio to test

You can either **upload your own** file, or let the notebook **generate a sample** so it
runs end-to-end with nothing to prepare. We'll do the generate-a-sample route here using
text-to-speech (the reverse of what we're building), then feed those clips into Whisper.

In [ ]:
from gtts import gTTS

# Short, natural sentences in each language (text-to-speech -> mp3).
hindi_text = 'नमस्ते, मेरा नाम आर्यन है और मुझे मशीन लर्निंग बहुत पसंद है।'
urdu_text  = 'السلام علیکم، میرا نام علی ہے اور میں مصنوعی ذہانت سیکھ رہا ہوں۔'

gTTS(hindi_text, lang='hi').save('sample_hindi.mp3')
gTTS(urdu_text,  lang='ur').save('sample_urdu.mp3')
print('Saved: sample_hindi.mp3, sample_urdu.mp3')

# Listen to them (optional)
from IPython.display import Audio, display
display(Audio('sample_hindi.mp3'))
display(Audio('sample_urdu.mp3'))

In [ ]:
# --- OPTION B: upload your OWN audio instead (uncomment in Colab) ---
# from google.colab import files
# uploaded = files.upload()          # pick a .mp3/.wav/.m4a/.ogg file
# sample_hindi = next(iter(uploaded))  # then use this path below

## 3️⃣  Approach 1 — OpenAI Whisper (simplest)

Load a model once, then call `.transcribe()`. Set `language='hi'` for Hindi or
`language='ur'` for Urdu. The output text comes back in the **native script**
(Devanagari for Hindi, Nastaʿlīq/Arabic script for Urdu).

In [ ]:
import whisper

# tiny < base < small < medium < large-v3   (bigger = more accurate, slower)
# 'small' is a good CPU-friendly demo; use 'large-v3' on a GPU for best results.
MODEL_SIZE = 'small'
model = whisper.load_model(MODEL_SIZE)
print(f'Loaded Whisper \'{MODEL_SIZE}\' on {model.device}')


def transcribe(audio_path, language):
    """Transcribe Hindi ('hi') or Urdu ('ur') speech to text in its native script."""
    result = model.transcribe(audio_path, language=language, task='transcribe', fp16=False)
    return result['text'].strip()


print('HINDI:', transcribe('sample_hindi.mp3', 'hi'))
print('URDU :', transcribe('sample_urdu.mp3', 'ur'))

### Bonus — auto-detect the language, or translate straight to English

In [ ]:
# 1) Auto-detect: omit `language=` and Whisper figures it out.
auto = model.transcribe('sample_hindi.mp3', fp16=False)
print('Detected language :', auto['language'])
print('Transcript        :', auto['text'].strip())

# 2) Speech translation: task='translate' returns ENGLISH text for any input language.
en = model.transcribe('sample_urdu.mp3', task='translate', fp16=False)
print('Urdu -> English   :', en['text'].strip())

## 4️⃣  Approach 2 — Whisper via 🤗 Transformers

The Hugging Face `pipeline` wraps **`openai/whisper-large-v3`** (the strongest general
multilingual Whisper). It auto-chunks **long audio** and can return **timestamps** — handy
for subtitles.

In [ ]:
import torch
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
dtype  = torch.float16 if torch.cuda.is_available() else torch.float32

asr = pipeline(
    'automatic-speech-recognition',
    model='openai/whisper-large-v3',   # multilingual: Hindi, Urdu + ~97 more
    torch_dtype=dtype,
    device=device,
    chunk_length_s=30,                 # split long audio into 30s windows automatically
)

out = asr(
    'sample_hindi.mp3',
    generate_kwargs={'language': 'hindi', 'task': 'transcribe'},  # use 'urdu' for Urdu
    return_timestamps=True,
)

print('FULL TEXT:', out['text'].strip())
print('\nSEGMENTS (for subtitles):')
for seg in out['chunks']:
    print(f"  {seg['timestamp']}  {seg['text'].strip()}")

## 5️⃣  Approach 3 — Fine-tuned Hindi / Urdu models (best accuracy)

General Whisper is good; models **fine-tuned on lots of Hindi/Urdu speech** are usually
**more accurate** on these languages. Just change the `model=` id. All of these are real,
public checkpoints on the Hugging Face Hub:

**Hindi** — `vasista22/whisper-hindi-large-v2` (Speech Lab, IIT Madras) · lighter: `vasista22/whisper-hindi-small`, `vasista22/whisper-hindi-medium`  
**Urdu**  — `kingabzpro/whisper-large-v3-urdu` · lighter: `Abdullah17/whisper-small-urdu`, `ihanif/whisper-medium-urdu`

In [ ]:
from transformers import pipeline
import torch
device = 0 if torch.cuda.is_available() else -1


def load_finetuned(model_id, lang_code):
    """Load a fine-tuned Whisper ASR pipeline and lock it to the given language."""
    pipe = pipeline('automatic-speech-recognition', model=model_id,
                    chunk_length_s=30, device=device)
    # Force the decoder to transcribe (not translate) in the target language.
    pipe.model.config.forced_decoder_ids = pipe.tokenizer.get_decoder_prompt_ids(
        language=lang_code, task='transcribe')
    return pipe


# --- Hindi (fine-tuned) ---
hindi_asr = load_finetuned('vasista22/whisper-hindi-large-v2', 'hi')
print('Hindi (fine-tuned):', hindi_asr('sample_hindi.mp3')['text'].strip())

# --- Urdu (fine-tuned) ---
urdu_asr = load_finetuned('kingabzpro/whisper-large-v3-urdu', 'ur')
print('Urdu  (fine-tuned):', urdu_asr('sample_urdu.mp3')['text'].strip())

## 6️⃣  Tips & practical notes

**Model size vs. accuracy** (pick based on your hardware):

| Size | Params | RAM/VRAM | Speed | Hindi/Urdu quality |
|------|--------|----------|-------|--------------------|
| `tiny` / `base` | 39M / 74M | ~1 GB | ⚡⚡⚡ | rough |
| `small` | 244M | ~2 GB | ⚡⚡ | decent |
| `medium` | 769M | ~5 GB | ⚡ | good |
| `large-v3` | 1.5B | ~10 GB | slow on CPU | **best** |

- **GPU strongly recommended** for `medium`/`large-v3`. On CPU, stick to `small`.
- **Long recordings**: Whisper works on 30-second windows internally. The Transformers
  `pipeline(..., chunk_length_s=30)` handles hours of audio automatically.
- **Native script vs. Roman**: output is in Devanagari/Urdu script. To get Roman/Latin
  (“Hinglish”) you'd transliterate afterwards (e.g. the `indic-transliteration` library).
- **Accuracy metric**: ASR quality is measured by **WER** (Word Error Rate) — lower is better.
- **Formats**: `.mp3`, `.wav`, `.m4a`, `.ogg`, `.flac` all work (ffmpeg decodes them).

## 🧰  Reusable helper — transcribe any file
Drop-in function you can copy into your own project.

In [ ]:
import whisper

_MODEL_CACHE = {}

def hindi_urdu_transcribe(audio_path, language='hi', size='small'):
    """Transcribe Hindi ('hi') or Urdu ('ur') audio to text.

    Args:
        audio_path: path to an audio file (mp3/wav/m4a/ogg/flac).
        language:   'hi' for Hindi, 'ur' for Urdu, or None to auto-detect.
        size:       whisper model size ('tiny'..'large-v3').
    Returns:
        Transcribed text in the native script.
    """
    if size not in _MODEL_CACHE:
        _MODEL_CACHE[size] = whisper.load_model(size)
    result = _MODEL_CACHE[size].transcribe(audio_path, language=language,
                                           task='transcribe', fp16=False)
    return result['text'].strip()


# Example:
print(hindi_urdu_transcribe('sample_hindi.mp3', language='hi'))
print(hindi_urdu_transcribe('sample_urdu.mp3',  language='ur'))

---
### ✔️ Summary
- **Whisper is multilingual** — it already understands Hindi & Urdu, no separate “install” of a
  different architecture required; you just select the language.
- **Approach 1** (`openai-whisper`) is the quickest start.
- **Approach 2** (🤗 Transformers `whisper-large-v3`) is best for long audio + timestamps.
- **Approach 3** (fine-tuned `vasista22` / `kingabzpro` checkpoints) gives the **highest
  accuracy** on Hindi/Urdu specifically.

**Next steps:** try your own recordings, compare model sizes, or fine-tune Whisper on your
own Hindi/Urdu data with the [`whisper-finetune`](https://github.com/vasistalodagala/whisper-finetune) recipe.